In [4]:
import pandas as pd
import numpy as np

# Load CSV (handles extra trailing commas)
df = pd.read_csv(
    "credit_card_unsupervised_transactions_random_count.csv",
    usecols=["customer_id", "transaction_id", "transaction_date",
             "transaction_amount", "merchant_category"]
)

# Basic cleaning
df.columns = df.columns.str.strip()
df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce")
df["transaction_amount"] = pd.to_numeric(df["transaction_amount"], errors="coerce")

# Drop invalid rows
df = df.dropna(subset=["customer_id", "transaction_amount", "merchant_category"])

print(df.head())


  customer_id transaction_id transaction_date  transaction_amount  \
0   CUST00000    TX000000001       2025-03-12           165352.27   
1   CUST00000    TX000000002       2025-02-22            80855.47   
2   CUST00000    TX000000003       2025-02-17            79085.63   
3   CUST00000    TX000000004       2025-05-11             6678.85   
4   CUST00000    TX000000005       2025-01-23            14279.63   

  merchant_category  
0            Dining  
1              Fuel  
2            Dining  
3            Dining  
4            Dining  


In [5]:
# Total transactions
txn_count = df.groupby("customer_id")["transaction_id"].count()

# Average transaction value
avg_txn_value = df.groupby("customer_id")["transaction_amount"].mean()

# Activity frequency (transactions per active day)
active_days = df.groupby("customer_id")["transaction_date"].nunique()
txn_per_day = txn_count / active_days

# Category ratios
category_pivot = (
    df.pivot_table(
        index="customer_id",
        columns="merchant_category",
        values="transaction_amount",
        aggfunc="sum",
        fill_value=0
    )
)

# Normalize category spend to ratios
category_ratio = category_pivot.div(category_pivot.sum(axis=1), axis=0)

# Final feature table
features = pd.concat(
    [txn_count, avg_txn_value, txn_per_day, category_ratio],
    axis=1
)

features.columns = (
    ["txn_count", "avg_txn_value", "txn_per_day"] +
    list(category_ratio.columns)
)

print(features.head())


             txn_count  avg_txn_value  txn_per_day    Dining  Entertainment  \
customer_id                                                                   
CUST00000           28   35003.705357     1.076923  0.369095       0.067042   
CUST00001            7    1102.082857     1.000000  0.000000       0.000000   
CUST00002           53    7392.142264     1.177778  0.269432       0.266483   
CUST00003           76     976.804342     1.245902  0.038870       0.011496   
CUST00004          136     319.861912     1.478261  0.140889       0.050544   

                 Fuel   Grocery  Online Shopping    Travel  Utilities  
customer_id                                                            
CUST00000    0.120086  0.047725         0.184616  0.202831   0.008605  
CUST00001    0.262808  0.030298         0.123392  0.567608   0.015893  
CUST00002    0.013546  0.041324         0.014701  0.327362   0.067152  
CUST00003    0.322347  0.316501         0.097979  0.090893   0.121915  
CUST00004    0

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)


In [7]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.90, random_state=42)  # keep 90% variance
pca_features = pca.fit_transform(scaled_features)

print("PCA Components:", pca.n_components_)


PCA Components: 7


In [8]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=20
)

clusters = kmeans.fit_predict(pca_features)

features["cluster_id"] = clusters


In [9]:
cluster_profile = features.groupby("cluster_id").mean()
print(cluster_profile)


             txn_count  avg_txn_value  txn_per_day    Dining  Entertainment  \
cluster_id                                                                    
0            32.553942   15507.750813     1.094667  0.266074       0.085497   
1            44.992145    1154.553536     1.125954  0.054405       0.029694   
2           113.360518     387.211827     1.353163  0.147862       0.051290   
3             8.143519    6039.488822     1.021768  0.079731       0.052928   
4             8.096552    5085.370982     1.022423  0.080942       0.515561   

                Fuel   Grocery  Online Shopping    Travel  Utilities  
cluster_id                                                            
0           0.053325  0.058452         0.105138  0.314569   0.116946  
1           0.322516  0.464055         0.040734  0.049528   0.039068  
2           0.252540  0.351936         0.099024  0.049104   0.048244  
3           0.058437  0.137662         0.520356  0.084820   0.066066  
4           0.077809

In [10]:
cluster_name_map = {
    0: "High txn count / Low avg value – Daily Spender",
    1: "Low txn count / High avg value – Premium User",
    2: "High Groceries/Fuel ratio – Household User",
    3: "High Travel & Dining – Lifestyle User",
    4: "Low Activity – Dormant User"
}

features["customer_segment"] = features["cluster_id"].map(cluster_name_map)


In [11]:
final_output = features.reset_index()[[
    "customer_id", "customer_segment"
]]

print(final_output.head())

# Save results
final_output.to_csv("customer_segments.csv", index=False)


  customer_id                                customer_segment
0   CUST00000  High txn count / Low avg value – Daily Spender
1   CUST00001  High txn count / Low avg value – Daily Spender
2   CUST00002  High txn count / Low avg value – Daily Spender
3   CUST00003   Low txn count / High avg value – Premium User
4   CUST00004      High Groceries/Fuel ratio – Household User
